In [ ]:
# Instalación de librerías necesarias
!pip install spacy pandas numpy tqdm regex

# Descargar modelo en español de spaCy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 64.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import re
import regex
import spacy
from tqdm import tqdm

# Cargar modelo de spaCy en español
nlp = spacy.load("es_core_news_sm")

# Acelerar apply con barra de progreso
tqdm.pandas()

In [ ]:
# Cargar dataset (ajusta la ruta si es necesario)
df = pd.read_csv("/content/IMDB Dataset SPANISH.csv")

# Ver primeras filas
df.head()

,Unnamed: 0,review_en,review_es,sentiment,sentimiento
0,0,One of the other reviewers has mentioned that ...,Uno de los otros críticos ha mencionado que de...,positive,positivo
1,1,A wonderful little production. The filming tec...,Una pequeña pequeña producción.La técnica de f...,positive,positivo
2,2,I thought this was a wonderful way to spend ti...,Pensé que esta era una manera maravillosa de p...,positive,positivo
3,3,Basically there's a family where a little boy ...,"Básicamente, hay una familia donde un niño peq...",negative,negativo
4,4,"Petter Mattei's ""Love in the Time of Money"" is...","El ""amor en el tiempo"" de Petter Mattei es una...",positive,positivo


In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
# 1. Eliminar URLs
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

# 2. Eliminar menciones y hashtags
def remove_mentions_hashtags(text):
    return re.sub(r'@\w+|#\w+', '', text)

# 3. Eliminar HTML
def remove_html(text):
    return re.sub(r'<.*?>', '', text)

# 4. Eliminar emojis
def remove_emojis(text):
    emoji_pattern = regex.compile(
        "[\p{Emoji_Presentation}\p{Extended_Pictographic}]",
        flags=regex.UNICODE
    )
    return emoji_pattern.sub(r'', text)

# 5. Eliminar elongaciones (muuuuy → muy)
def remove_elongations(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

# 6. Eliminar puntuación
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

# 7. Minúsculas
def to_lowercase(text):
    return text.lower()

# 10. Limpiar espacios
def clean_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

<>:16: SyntaxWarning: invalid escape sequence '\p'
<>:16: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_26523/1673865323.py:16: SyntaxWarning: invalid escape sequence '\p'
  "[\p{Emoji_Presentation}\p{Extended_Pictographic}]",


In [ ]:
def lemmatize_and_remove_stopwords(text):
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_space
    ]
    return " ".join(tokens)

In [ ]:
def preprocess_text(text):
    if pd.isnull(text):
        return ""

    text = remove_urls(text)
    text = remove_mentions_hashtags(text)
    text = remove_html(text)
    text = remove_emojis(text)
    text = remove_elongations(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    text = lemmatize_and_remove_stopwords(text)
    text = clean_spaces(text)

    return text

In [ ]:
# Aplicar limpieza con barra de progreso
df['review_clean'] = df['review_es'].progress_apply(preprocess_text)

# Ver resultados
df[['review_es', 'review_clean']].head()

100%|██████████| 50000/50000 [30:31<00:00, 27.30it/s]


,review_es,review_clean
0,Uno de los otros críticos ha mencionado que de...,crítico mencionar 1 oz episodio enganchado raz...
1,Una pequeña pequeña producción.La técnica de f...,pequeño pequeño producciónla técnico filmación...
2,Pensé que esta era una manera maravillosa de p...,pensar maravilloso pasar tiempo semana verano ...
3,"Básicamente, hay una familia donde un niño peq...",básicamente familia niño pequeño jake pensar z...
4,"El ""amor en el tiempo"" de Petter Mattei es una...",amor tiempo petter mattei película visualmente...


In [ ]:
df.head()

,Unnamed: 0,review_en,review_es,sentiment,sentimiento,review_clean
0,0,One of the other reviewers has mentioned that ...,Uno de los otros críticos ha mencionado que de...,1,positivo,crítico mencionar 1 oz episodio enganchado raz...
1,1,A wonderful little production. The filming tec...,Una pequeña pequeña producción.La técnica de f...,1,positivo,pequeño pequeño producciónla técnico filmación...
2,2,I thought this was a wonderful way to spend ti...,Pensé que esta era una manera maravillosa de p...,1,positivo,pensar maravilloso pasar tiempo semana verano ...
3,3,Basically there's a family where a little boy ...,"Básicamente, hay una familia donde un niño peq...",0,negativo,básicamente familia niño pequeño jake pensar z...
4,4,"Petter Mattei's ""Love in the Time of Money"" is...","El ""amor en el tiempo"" de Petter Mattei es una...",1,positivo,amor tiempo petter mattei película visualmente...


In [ ]:
df.to_csv("imdb_clean.csv", index=False)

In [ ]:
!pip install mlflow
!pip install tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.2 MB/s eta 0:00:00


In [ ]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import mlflow
import mlflow.tensorflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import tensorflow as tf
from tensorflow import keras
import gensim.downloader as api

def main():
    mlflow.set_tracking_uri("http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/")
    mlflow.set_experiment("baseline_tfidf_lr")

    csv_path = "/content/imdb_clean.csv"
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['review_clean', 'sentiment'])

    X = df['review_clean'].astype(str).values
    y = df['sentiment'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # ─── Hiperparámetros (mismos que paso 3) ────────────────────────────────
    MAX_TOKENS    = 20000
    SEQUENCE_LEN  = 200
    EMBEDDING_DIM = 300   # Word2Vec google-news usa 300 dimensiones
    HIDDEN_UNITS  = [128, 64]
    DROPOUT_RATE  = 0.4
    EPOCHS        = 10
    BATCH_SIZE    = 64
    # ────────────────────────────────────────────────────────────────────────

    # TextVectorization (requerido por el enunciado para redes neuronales)
    vectorizer = keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_mode='int',
        output_sequence_length=SEQUENCE_LEN
    )
    vectorizer.adapt(X_train)

    # Convertir texto a secuencias de enteros
    X_train_seq = vectorizer(X_train).numpy()
    X_test_seq  = vectorizer(X_test).numpy()

    # ── Cargar Word2Vec y construir matriz de embeddings ─────────────────────
    print("Cargando Word2Vec (google-news-300)... esto puede tardar unos minutos.")
    w2v_model = api.load("word2vec-google-news-300")
    print("Word2Vec cargado exitosamente.")

    vocab = vectorizer.get_vocabulary()
    embedding_matrix = np.zeros((MAX_TOKENS, EMBEDDING_DIM))
    found = 0
    for idx, word in enumerate(vocab):
        if idx >= MAX_TOKENS:
            break
        if word in w2v_model:
            embedding_matrix[idx] = w2v_model[word]
            found += 1
    print(f"Palabras encontradas en Word2Vec: {found}/{min(len(vocab), MAX_TOKENS)}")

    # Construcción del modelo con embeddings CONGELADOS
    model = keras.Sequential([
        keras.layers.Embedding(
            input_dim=MAX_TOKENS,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            input_length=SEQUENCE_LEN,
            trainable=False,           # ← CONGELADO
            name="word2vec_embedding_frozen"
        ),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(HIDDEN_UNITS[0], activation='relu'),
        keras.layers.Dropout(DROPOUT_RATE),
        keras.layers.Dense(HIDDEN_UNITS[1], activation='relu'),
        keras.layers.Dropout(DROPOUT_RATE),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    with mlflow.start_run(run_name="word2vec_frozen"):
        mlflow.set_tag("username", "Andres")
        mlflow.set_tag("step", "4_word2vec_frozen")
        mlflow.set_tag("embedding_type", "Word2Vec")
        mlflow.set_tag("trainable", "False")

        # Log parámetros
        mlflow.log_param("max_tokens", MAX_TOKENS)
        mlflow.log_param("sequence_len", SEQUENCE_LEN)
        mlflow.log_param("embedding_dim", EMBEDDING_DIM)
        mlflow.log_param("hidden_units", str(HIDDEN_UNITS))
        mlflow.log_param("dropout_rate", DROPOUT_RATE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("optimizer", "adam")
        mlflow.log_param("embedding_source", "word2vec-google-news-300")
        mlflow.log_param("embeddings_trainable", False)

        # Entrenamiento
        print("Training model...")
        history = model.fit(
            X_train_seq, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.1,
            verbose=1
        )

        # Evaluación
        print("Evaluating model...")
        y_prob = model.predict(X_test_seq)
        y_pred = (y_prob >= 0.5).astype(int).flatten()

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='weighted')

        print(f"Accuracy : {acc:.4f}")
        print(f"F1 Score : {f1:.4f}")

        # Log métricas finales
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", acc)

        # Log métricas por época
        for epoch, (loss, val_loss, acc_e, val_acc) in enumerate(zip(
            history.history['loss'],
            history.history['val_loss'],
            history.history['accuracy'],
            history.history['val_accuracy']
        )):
            mlflow.log_metric("train_loss", loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("train_accuracy", acc_e, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        # Log modelo
        mlflow.tensorflow.log_model(model, "model")
        print("Model logged to MLflow successfully.")

if __name__ == "__main__":
    main()


Loading data from /content/imdb_clean.csv...
Cargando Word2Vec (google-news-300)... esto puede tardar unos minutos.
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Word2Vec cargado exitosamente.
Palabras encontradas en Word2Vec: 8649/20000


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word2vec_embedding_frozen       │ ?                      │     6,000,000 │
│ (Embedding)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,000,000 (22.89 MB)

Training model...
Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.5825 - loss: 0.6711 - val_accuracy: 0.6755 - val_loss: 0.6208
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.6686 - loss: 0.6136 - val_accuracy: 0.6643 - val_loss: 0.6033
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.6873 - loss: 0.5930 - val_accuracy: 0.6950 - val_loss: 0.5816
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.6929 - loss: 0.5819 - val_accuracy: 0.7100 - val_loss: 0.5709
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.6971 - loss: 0.5779 - val_accuracy: 0.7095 - val_loss: 0.5683
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.7026 - loss: 0.5722 - val_accuracy: 0.7160 - val_loss: 0.5627
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.7052 - loss: 0.5702 - val_accuracy: 0.7110 - val_loss: 0.5691
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.7085

2026/04/14 02:49:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 02:49:22 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


Model logged to MLflow successfully.
🏃 View run word2vec_frozen at: http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/#/experiments/6/runs/491e695fdf0c4e0fb98768298e83a8e4
🧪 View experiment at: http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/#/experiments/6


In [ ]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import mlflow
import mlflow.tensorflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import tensorflow as tf
from tensorflow import keras
import gensim.downloader as api

def main():
    mlflow.set_tracking_uri("http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/")
    mlflow.set_experiment("baseline_tfidf_lr")

    csv_path = "/content/imdb_clean.csv"
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['review_clean', 'sentiment'])

    X = df['review_clean'].astype(str).values
    y = df['sentiment'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # ─── Hiperparámetros (mismos que paso 3) ────────────────────────────────
    MAX_TOKENS    = 20000
    SEQUENCE_LEN  = 200
    EMBEDDING_DIM = 300   # Word2Vec google-news usa 300 dimensiones
    HIDDEN_UNITS  = [128, 64]
    DROPOUT_RATE  = 0.4
    EPOCHS        = 10
    BATCH_SIZE    = 64
    # ────────────────────────────────────────────────────────────────────────

    # TextVectorization (requerido por el enunciado para redes neuronales)
    vectorizer = keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_mode='int',
        output_sequence_length=SEQUENCE_LEN
    )
    vectorizer.adapt(X_train)

    # Convertir texto a secuencias de enteros
    X_train_seq = vectorizer(X_train).numpy()
    X_test_seq  = vectorizer(X_test).numpy()

    # ── Cargar Word2Vec y construir matriz de embeddings ─────────────────────
    print("Cargando Word2Vec (google-news-300)... esto puede tardar unos minutos.")
    w2v_model = api.load("word2vec-google-news-300")
    print("Word2Vec cargado exitosamente.")

    vocab = vectorizer.get_vocabulary()
    embedding_matrix = np.zeros((MAX_TOKENS, EMBEDDING_DIM))
    found = 0
    for idx, word in enumerate(vocab):
        if idx >= MAX_TOKENS:
            break
        if word in w2v_model:
            embedding_matrix[idx] = w2v_model[word]
            found += 1
    print(f"Palabras encontradas en Word2Vec: {found}/{min(len(vocab), MAX_TOKENS)}")

    # Construcción del modelo con embeddings AJUSTABLES
    model = keras.Sequential([
        keras.layers.Embedding(
            input_dim=MAX_TOKENS,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            input_length=SEQUENCE_LEN,
            trainable=True,            # ← AJUSTABLE
            name="word2vec_embedding_trainable"
        ),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(HIDDEN_UNITS[0], activation='relu'),
        keras.layers.Dropout(DROPOUT_RATE),
        keras.layers.Dense(HIDDEN_UNITS[1], activation='relu'),
        keras.layers.Dropout(DROPOUT_RATE),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    with mlflow.start_run(run_name="word2vec_trainable"):
        mlflow.set_tag("username", "Andres")
        mlflow.set_tag("step", "4_word2vec_trainable")
        mlflow.set_tag("embedding_type", "Word2Vec")
        mlflow.set_tag("trainable", "True")

        # Log parámetros
        mlflow.log_param("max_tokens", MAX_TOKENS)
        mlflow.log_param("sequence_len", SEQUENCE_LEN)
        mlflow.log_param("embedding_dim", EMBEDDING_DIM)
        mlflow.log_param("hidden_units", str(HIDDEN_UNITS))
        mlflow.log_param("dropout_rate", DROPOUT_RATE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("optimizer", "adam")
        mlflow.log_param("embedding_source", "word2vec-google-news-300")
        mlflow.log_param("embeddings_trainable", True)

        # Entrenamiento
        print("Training model...")
        history = model.fit(
            X_train_seq, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.1,
            verbose=1
        )

        # Evaluación
        print("Evaluating model...")
        y_prob = model.predict(X_test_seq)
        y_pred = (y_prob >= 0.5).astype(int).flatten()

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='weighted')

        print(f"Accuracy : {acc:.4f}")
        print(f"F1 Score : {f1:.4f}")

        # Log métricas finales
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", acc)

        # Log métricas por época
        for epoch, (loss, val_loss, acc_e, val_acc) in enumerate(zip(
            history.history['loss'],
            history.history['val_loss'],
            history.history['accuracy'],
            history.history['val_accuracy']
        )):
            mlflow.log_metric("train_loss", loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("train_accuracy", acc_e, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        # Log modelo
        mlflow.tensorflow.log_model(model, "model")
        print("Model logged to MLflow successfully.")

if __name__ == "__main__":
    main()


Loading data from /content/imdb_clean.csv...
Cargando Word2Vec (google-news-300)... esto puede tardar unos minutos.
Word2Vec cargado exitosamente.
Palabras encontradas en Word2Vec: 8649/20000


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word2vec_embedding_trainable    │ ?                      │     6,000,000 │
│ (Embedding)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,000,000 (22.89 MB)

 Trainable params: 6,000,000 (22.89 MB)

 Non-trainable params: 0 (0.00 B)

Training model...
Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 56s 95ms/step - accuracy: 0.6510 - loss: 0.5910 - val_accuracy: 0.8213 - val_loss: 0.4264
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 51s 91ms/step - accuracy: 0.8459 - loss: 0.3565 - val_accuracy: 0.8610 - val_loss: 0.3557
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 81s 91ms/step - accuracy: 0.8778 - loss: 0.2921 - val_accuracy: 0.7912 - val_loss: 0.4891
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - accuracy: 0.8964 - loss: 0.2539 - val_accuracy: 0.8670 - val_loss: 0.3358
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 51s 90ms/step - accuracy: 0.9053 - loss: 0.2339 - val_accuracy: 0.8677 - val_loss: 0.3597
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.9056 - loss: 0.2271 - val_accuracy: 0.8092 - val_loss: 0.4603
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 51s 90ms/step - accuracy: 0.9223 - loss: 0.1963 - val_accuracy: 0.8655 - val_loss: 0.4196
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.9314

2026/04/14 03:03:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 03:03:13 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run word2vec_trainable at: http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/#/experiments/6/runs/afa81c4c0a044e21bde912fbaa2be3a2
🧪 View experiment at: http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/#/experiments/6


MlflowException: API request to http://ec2-3-94-210-185.compute-1.amazonaws.com:5000/api/2.0/mlflow-artifacts/artifacts/6/models/m-7932d6a79e784874b1e6e8dbbe4855f6/artifacts/data/model.keras failed with exception HTTPConnectionPool(host='ec2-3-94-210-185.compute-1.amazonaws.com', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/6/models/m-7932d6a79e784874b1e6e8dbbe4855f6/artifacts/data/model.keras (Caused by ResponseError('too many 500 error responses'))